# Theta-sweep pipeline

This notebook defines every pipeline function inline — everything except `Config`, the NWB reader, and the
plotters. Run it cell by cell to see what each function does. (`decode` is the slow stage, ~1–2 min.)

### Imports — library primitives + the non-analysis helpers only
`numpy`, the scipy filters the functions use, and `Config` / `load_session` / `Result` / `Session` /
`plot_*`. No `theta_cycles`, `decode`, `extract_sweeps`, etc. — those are defined below.

In [ ]:
import sys, numpy as np, matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation, gaussian_filter, gaussian_filter1d
from scipy.signal import butter, filtfilt, hilbert
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
sys.path.insert(0, "/Users/minhphan/Documents/Reproduce-Theta-Sweeps-from-HexMaze-dataset")
from hexmaze_sweeps import Config, load_session
from hexmaze_sweeps.data import Result, Session
from hexmaze_sweeps.plotting import plot_sweeps, plot_all_sweeps

NWB   = "/Users/minhphan/Documents/Internship_HMNeuron/nwb/Rat6_20260629.nwb"
NODES = "/Users/minhphan/Documents/Reproduce-Theta-Sweeps-from-HexMaze-dataset/node_list_new.csv"


### Config — the knobs. Edit and re-run, then re-run stages 3+.

In [ ]:
cfg = Config(decoder="bayes", bayes_prior="uniform", pv_smooth_bins=5.0,
             min_active_cells=1, max_lowpass_error_cm=50)
print(f"decoder={cfg.decoder}  smoothing={cfg.pv_smooth_bins*10:.0f} ms  spatial_bin={cfg.spatial_bin_cm}cm")


## 1. `load_session` — turn one NWB into a `Session`

Reads a single recording and assembles clean, time-aligned arrays; nothing is analysed yet. It **selects
the units** you asked for (`good` pyramidal), **bins each unit's spike times into 10 ms bins** →
`spike_counts (bins × units)`, reads the **tracking** and interpolates x/y onto those bins, computes
**speed** and **head direction from travel** (DLC head-tracking is unusable here), pulls **one LFP channel**
for theta, and loads the **maze nodes**. Returns a `Session` + open handle `io`.

*(This one stays imported — it's NWB I/O, not analysis. `print(inspect.getsource(load_session))` to read it.)*

In [ ]:
import inspect
sess, io = load_session(NWB, NODES, cfg)
print(f"units {sess.n_units} | bins {len(sess.bin_centers_s)} "
      f"({len(sess.bin_centers_s)*cfg.bin_s/60:.1f} min) | spike_counts {sess.spike_counts.shape}")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(sess.track_x_px, sess.track_y_px, lw=0.4, c="0.5"); ax[0].invert_yaxis()
ax[0].set_aspect("equal"); ax[0].set_title("trajectory")
ax[1].hist(sess.speed_px_s/cfg.px_per_cm, bins=80, range=(0,80)); ax[1].axvline(cfg.speed_sweep_cm_s, c="r", ls="--")
ax[1].set_title("speed (cm/s); red = run gate"); plt.tight_layout(); plt.show()


## 2. `theta_cycles` — where each theta cycle begins

Returns the **10 ms bin index at which each theta cycle starts**:
1. **A theta phase per bin** — band-pass the LFP to **5–10 Hz** (Butterworth), **Hilbert** for the
   instantaneous phase (−π→π once per cycle), interpolate onto the bins. (`_theta_phase_from_population`
   is the paper's PCA-on-spikes alternative.)
2. **Re-zero at the theta trough** — anchor phase 0 to the point where the **population fires least**.
3. **Mark cycle starts** at upward zero-crossings of that phase.
4. **Keep only 0.08–0.22 s cycles** (≈4.5–12.5 Hz).

In [ ]:
from __future__ import annotations
# ==== DEFINE: _theta_phase_from_lfp, _theta_phase_from_population, theta_cycles   (this is the real code — edit + re-run to change the pipeline) ====

def _theta_phase_from_lfp(session: Session, config: Config) -> np.ndarray:
    
    """Theta phase per time bin, band-passed from the field potential."""
    
    nyquist_hz = session.lfp_rate_hz / 2
    filter_b, filter_a = butter(2, [config.theta_band_hz[0] / nyquist_hz,
                                    config.theta_band_hz[1] / nyquist_hz], "band") # 5-10 Hz
    lfp_phase = np.angle(hilbert(filtfilt(filter_b, filter_a, session.lfp_theta_channel))) #instaneous phase
    lfp_t_s = np.arange(len(lfp_phase)) / session.lfp_rate_hz # cycles extracted

    # np.interp repeats its last value rather than extrapolating, which would freeze
    # the phase and destroy every cycle past the end of the LFP. Refuse instead.
    
    coverage = lfp_t_s[-1] / session.bin_centers_s[-1]
    if coverage < 0.95:
        raise ValueError(
            f"LFP covers only {coverage * 100:.0f}% of the session "
            f"({lfp_t_s[-1]:.0f} s of {session.bin_centers_s[-1]:.0f} s). "
            f"Set Config.lfp_rate_hz explicitly, or use theta_source='pca'.")

    phase = np.interp(session.bin_centers_s, lfp_t_s, np.unwrap(lfp_phase))
    return (phase + np.pi) % (2 * np.pi) - np.pi


def _theta_phase_from_population(session: Session, config: Config) -> np.ndarray:
    """Theta phase per time bin, reconstructed from spiking (the paper's method).

    Band-passed spike counts trace a circle in their first two principal components
    once per theta cycle; the angle around that circle is the phase.
    """
    nyquist_hz = config.bin_rate_hz / 2
    filter_b, filter_a = butter(2, [config.theta_band_hz[0] / nyquist_hz,
                                    config.theta_band_hz[1] / nyquist_hz], "band")
    filtered = filtfilt(filter_b, filter_a, session.spike_counts, axis=0)

    centered = filtered - filtered.mean(0)
    svd_u, svd_s, _ = np.linalg.svd(centered, full_matrices=False)
    pc_scores = svd_u[:, :2] * svd_s[:2]

    phase = np.arctan2(pc_scores[:, 1], pc_scores[:, 0])

    # A principal component's sign is arbitrary, so the circle may be traced backwards.
    if np.median(np.diff(np.unwrap(phase))) < 0:
        phase = -phase
    return phase


def theta_cycles(session: Session, config: Config) -> np.ndarray:
    """
    Bin index at which each theta cycle starts.

    Phase is only defined up to an offset, so it is re-zeroed at the phase where the
    population fires least (the paper's convention). Cycles then begin at upward
    zero-crossings of the re-zeroed phase.
    
    """
    if config.theta_source == "lfp" and session.lfp_theta_channel is not None:
        phase = _theta_phase_from_lfp(session, config)
    else:
        phase = _theta_phase_from_population(session, config)

    # --- find the phase at which the population fires least, and call it zero --
    population_spikes = session.spike_counts.sum(1)
    n_phase_bins = 60 

    phase_bin = (((phase + np.pi) / (2 * np.pi)) * n_phase_bins).astype(int) % n_phase_bins # conv to [0;2pi]
    mean_rate_per_phase_bin = np.array([
        population_spikes[phase_bin == k].mean() if np.any(phase_bin == k) else np.inf
        for k in range(n_phase_bins)])

    quietest_bin = np.argmin(mean_rate_per_phase_bin) # min fire
    min_firing_phase = (quietest_bin + 0.5) / n_phase_bins * 2 * np.pi - np.pi 
    phase = (phase - min_firing_phase + np.pi) % (2 * np.pi) - np.pi # revert

    # --- cycles start at upward zero-crossings; drop non-theta durations -------
    onsets = np.where((phase[:-1] < 0) & (phase[1:] >= 0))[0] + 1

    # The last onset is dropped: there is no following onset to measure it against.
    duration_s = np.diff(onsets) * config.bin_s
    is_theta = (duration_s >= config.cycle_min_s) & (duration_s <= config.cycle_max_s)
    return onsets[:-1][is_theta]


In [ ]:
onsets = theta_cycles(sess, cfg)
print(f"cycles {len(onsets)} | mean {np.diff(onsets).mean()*cfg.bin_s*1000:.0f} ms")
w0,w1=2000,2300; plt.figure(figsize=(11,2.5))
plt.plot(sess.bin_centers_s[w0:w1], gaussian_filter1d(sess.spike_counts[w0:w1].sum(1).astype(float),1.5))
for o in onsets[(onsets>=w0)&(onsets<w1)]: plt.axvline(sess.bin_centers_s[o], c="r", alpha=0.3)
plt.title("population firing; red = onsets"); plt.show()


## 3. `rate_maps` — each cell's place map

Where each cell fires. Uses only **running** bins (> 5 cm/s), tiles the arena into **5 cm squares**, sums
each cell's **spikes** and the **time spent** per square, smooths both (**7.5 cm** Gaussian), then
**spikes ÷ occupancy** = Hz per cell per square. Squares visited < 0.25 s are dropped. Output: `tuning_hz`
(the template the decoder matches against) + square coords `bx, by` + occupancy `occ`.

In [ ]:
from __future__ import annotations
# ==== DEFINE: rate_maps   (this is the real code — edit + re-run to change the pipeline) ====

def rate_maps(session: Session, config: Config, include_bins: np.ndarray | None = None):
    """Build the tuning curves the decoder compares against.

    Rates are returned in spikes per second then fed to `prepare_tuning` 

    Args:
        include_bins: optional boolean mask over time bins. When given, only those bins
            contribute to the maps -- this is how `cross_validate_smoothing` holds a fold
            out. The spatial grid still spans the whole session, so bin centres stay
            comparable across folds; only which bins are counted changes.

    Returns:
        tuning_hz: (n_positions, n_units), each cell's firing rate at each position.
        bin_center_x_px, bin_center_y_px: (n_positions,).
        occupancy_s: (n_positions,) -- seconds spent at each, for the Bayesian prior.
    """
    bin_size_px = config.px(config.spatial_bin_cm)
    smooth_sigma_bins = config.px(config.rate_smooth_cm) / bin_size_px

    is_running = (session.speed_px_s > config.px(config.speed_spatial_cm_s)) \
        & np.isfinite(session.speed_px_s)
    if include_bins is not None:
        is_running = is_running & include_bins

    # --- diving maze into smaller grid ---------------------------------------------
    x_edges_px = np.arange(np.nanmin(session.track_x_px),
                           np.nanmax(session.track_x_px) + bin_size_px, bin_size_px)
    y_edges_px = np.arange(np.nanmin(session.track_y_px),
                           np.nanmax(session.track_y_px) + bin_size_px, bin_size_px)
    n_x_bins, n_y_bins = len(x_edges_px) - 1, len(y_edges_px) - 1

    # --- how long the animal spent in each square ------------------------------
    occupancy_s = np.histogram2d(session.track_x_px[is_running], session.track_y_px[is_running],
                                 bins=[x_edges_px, y_edges_px])[0] * config.bin_s
    occupancy_smoothed = gaussian_filter(occupancy_s, smooth_sigma_bins, mode="constant")

    # --- firing rate = smoothed spike count / smoothed time spent --------------
    maps = np.zeros((session.n_units, n_x_bins, n_y_bins), np.float32)
    for unit in range(session.n_units):
        spikes_while_running = session.spike_counts[:, unit] * is_running
        spike_map = gaussian_filter(
            np.histogram2d(session.track_x_px, session.track_y_px,
                           bins=[x_edges_px, y_edges_px], weights=spikes_while_running)[0],
            smooth_sigma_bins, mode="constant")
        with np.errstate(divide="ignore", invalid="ignore"):
            maps[unit] = np.where(occupancy_smoothed > 0.01, spike_map / occupancy_smoothed, 0.0)

    # --- which positions the decoder may choose from ---------------------------
    # Bins the animal visited, plus a margin of unvisited space around them so a sweep can
    # leave the travelled path. Their rate comes from the smoothing above, which carries
    # each cell's tuning a little way past the edge of where the animal actually went.
    visited = occupancy_s > config.min_occupancy_s

    if config.unvisited_margin_cm > 0:
        margin_bins = int(round(config.unvisited_margin_cm / config.spatial_bin_cm))
        visited = binary_dilation(visited, iterations=margin_bins)

        # Keep only bins where the rate maps are actually defined. Below this the loop above
        # sets every cell's rate to exactly zero, so the bin can never be decoded -- but it
        # would still count towards the decoder's 99th-percentile threshold and drag it
        # down, smearing the centroid across the maze.
        visited &= occupancy_smoothed > 0.01

    visited = visited.ravel()
    bin_center_x_px = (0.5 * (x_edges_px[:-1] + x_edges_px[1:]))[:, None].repeat(n_y_bins, 1).ravel()[visited]
    bin_center_y_px = (0.5 * (y_edges_px[:-1] + y_edges_px[1:]))[None, :].repeat(n_x_bins, 0).ravel()[visited]

    tuning_hz = maps.reshape(session.n_units, -1)[:, visited].T
    return tuning_hz, bin_center_x_px, bin_center_y_px, occupancy_s.ravel()[visited]


In [ ]:
tun_hz, bx, by, occ = rate_maps(sess, cfg)
print("tuning_hz", tun_hz.shape, "| visited bins", len(bx), "| peak", f"{tun_hz.max():.0f} Hz")
order = np.argsort(-tun_hz.max(0)); fig, axes = plt.subplots(2,5, figsize=(14,5.4))
for ax,u in zip(axes.flat, order[:10]):
    ax.scatter(bx,by,c=tun_hz[:,u],s=12,cmap="viridis"); ax.invert_yaxis(); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f"cell {u}", fontsize=8)
fig.suptitle("rate maps (10 most active)"); plt.tight_layout(); plt.show()


## 4. `prepare_tuning` + `prepare_log_prior`

Reshape the maps for the decoder. `prepare_tuning`: Bayes keeps raw Hz (floors zeros so `log` is finite);
PV mean-normalises each cell (match the *pattern*, not absolute rate). `prepare_log_prior`: `log P(x)`
over positions — flat (`None`) unless you pick the occupancy prior.

In [ ]:
from __future__ import annotations
# ==== DEFINE: prepare_tuning, prepare_log_prior   (this is the real code — edit + re-run to change the pipeline) ====

def prepare_tuning(tuning_hz: np.ndarray, config: Config) -> np.ndarray:
    """Rate maps -> what the decoder actually correlates or integrates against.

    The two decoders want different things from the same rate maps, and giving either the
    other's form is silently wrong rather than loud:

    * PV correlation divides each cell by its own mean rate (`decodePv.m` line 24), so that
      a high-firing cell cannot dominate a correlation taken ACROSS cells. This is a
      per-cell rescaling, and Pearson r across cells is not invariant to it, so it matters.
    * Bayesian reconstruction needs the rates themselves, in spikes per second, because the
      Poisson likelihood is a statement about how many spikes a rate produces in `bin_s`.
      Mean-normalising first would make `exp(-tau * sum(f))` meaningless.
    """
    if config.decoder == "bayes":
        return np.maximum(tuning_hz, config.bayes_min_rate_hz)
    return tuning_hz / (tuning_hz.mean(0, keepdims=True) + 1e-9)


def prepare_log_prior(occupancy_s: np.ndarray, config: Config) -> np.ndarray | None:
    """log P(x) for the Bayesian decoder. None means a flat prior (and for PV, unused)."""
    if config.decoder != "bayes" or config.bayes_prior != "occupancy":
        return None

    prior = occupancy_s / occupancy_s.sum()
    return np.log(prior + 1e-12)


In [ ]:
tun = prepare_tuning(tun_hz, cfg); lp = prepare_log_prior(occ, cfg)
print("tuning", tun.shape, "| log_prior", "None (uniform)" if lp is None else lp.shape)


## 5–6. The decoder + its chance level

**`decode`** answers, for every 10 ms bin, *where does the population say the animal is?* It smooths the
spike counts over ~50 ms, then scores every position — **Bayes** = Poisson probability of the spikes given
the rate maps (`log P(x|n)=Σ nᵢ log fᵢ(x) − τ Σ fᵢ(x) + prior`), **PV** = correlation with each position's
template — and reduces that score map to one continuous point with a **weighted centroid** of the top
scores (`_thresholded_centroid`), not a hard argmax.

**`shuffle_threshold`** is the chance level: slide each cell's spikes in time (keep rate + rhythm, destroy
place coding), decode, take the 99th-percentile score. A bin's decode must beat it to be trusted.

The cell below **defines the whole decoder**; the next defines `decode` and `shuffle_threshold`.

In [ ]:
from __future__ import annotations
# ==== DEFINE: decoder core (PV + Bayes + centroid readout)   (this is the real code — edit + re-run to change the pipeline) ====

def _correlate_across_units(activity: np.ndarray, tuning_curves: np.ndarray) -> np.ndarray:
    """Pearson r across cells: (n_time_bins, n_units) x (n_positions, n_units) -> (T, P)."""
    activity_z = (activity - activity.mean(1, keepdims=True)) / (activity.std(1, keepdims=True) + 1e-12)
    tuning_z = (tuning_curves - tuning_curves.mean(1, keepdims=True)) / (tuning_curves.std(1, keepdims=True) + 1e-12)
    return (activity_z @ tuning_z.T) / activity.shape[1]


def _bayes_posterior(activity, tuning_hz, config: Config, log_prior=None) -> np.ndarray:
    """Poisson posterior over position, one row per time bin. (T, n_units) -> (T, P).

    The standard rate-map reconstruction (Zhang et al. 1998; the same formulation
    pynapple's `decode_2d` uses), which is what the paper's Methods text describes even
    though the released code ships the PV-correlation decoder instead:

        log P(x | n)  =  sum_i n_i log f_i(x)  -  tau sum_i f_i(x)  +  log P(x)  +  const

    `const` absorbs `-sum_i log(n_i!)`, which does not depend on position and therefore
    cannot move the decoded position. That is what lets the SMOOTHED spike counts the PV
    decoder uses be passed in here unchanged: they are not integers, so the factorial term
    would be ill-defined, but it drops out of the posterior anyway.
    """
    log_likelihood = (activity @ np.log(tuning_hz).T
                      - config.bin_s * tuning_hz.sum(1)[None, :])

    if log_prior is not None:
        log_likelihood = log_likelihood + log_prior[None, :]

    # Subtract the row max before exponentiating, or 83 cells' worth of log-rates underflow.
    log_likelihood -= log_likelihood.max(1, keepdims=True)
    posterior = np.exp(log_likelihood)
    return posterior / posterior.sum(1, keepdims=True)


def decoder_scores(activity, tuning, config: Config, log_prior=None) -> np.ndarray:
    """(n_time_bins, n_units) -> (n_time_bins, n_positions). Higher means a better match.

    A correlation for `decoder="pv"`, a posterior probability for `decoder="bayes"`.
    Everything downstream -- the centroid, the shuffle, the anchor -- only asks that the
    score be higher where the position fits better, so it does not care which it is.
    """
    if config.decoder == "bayes":
        return _bayes_posterior(activity, tuning, config, log_prior)
    return _correlate_across_units(activity, tuning)


def _bin_distance_matrix_px(bin_center_x_px, bin_center_y_px) -> np.ndarray:
    """Distance between every pair of position bins. Computed once, reused per chunk."""
    return np.hypot(bin_center_x_px[:, None] - bin_center_x_px[None, :],
                    bin_center_y_px[:, None] - bin_center_y_px[None, :]).astype(np.float32)


def _thresholded_centroid(scores, bin_center_x_px, bin_center_y_px, config,
                          bin_distance_px=None):
    """Turn a map of decoder scores into one decoded position per time bin.

    A weighted average of the good positions, rather than the single best one, which
    would quantise onto the grid (processDec, inside runPvPosDecoding.m). A position
    counts as good if it is among the top few percent anywhere, or lies close to the peak.

    Returns:
        (decoded position per bin, peak score per bin).
    """
    if bin_distance_px is None:
        bin_distance_px = _bin_distance_matrix_px(bin_center_x_px, bin_center_y_px)

    peak_bin = scores.argmax(1)
    peak_score = scores[np.arange(len(scores)), peak_bin]

    n_positions = scores.shape[1]
    kth = int(np.clip(round(config.centroid_percentile / 100.0 * (n_positions - 1)),
                      0, n_positions - 1))
    threshold = np.partition(scores, kth, axis=1)[:, kth][:, None]

    weights = scores.copy()
    is_far_from_peak = bin_distance_px[peak_bin] > config.px(config.centroid_radius_cm)
    weights[(scores < threshold) & is_far_from_peak] = 0.0

    # `processDec` stops here, so by default so do we -- which leaves a negative weight on
    # any position that is anti-correlated but close to the peak, dragging the centroid
    # away from it. See `Config.clip_negative_weights`.
    if config.clip_negative_weights:
        np.clip(weights, 0, None, out=weights)

    weight_sum = weights.sum(1)
    weight_sum[weight_sum == 0] = np.nan            # nothing matched
    centroid = np.stack([(weights * bin_center_x_px).sum(1) / weight_sum,
                         (weights * bin_center_y_px).sum(1) / weight_sum], 1)
    return centroid, peak_score


In [ ]:
from __future__ import annotations
# ==== DEFINE: decode, shuffle_threshold   (this is the real code — edit + re-run to change the pipeline) ====

def decode(session, config, tuning, bin_center_x_px, bin_center_y_px, log_prior=None,
           chunk_size=8000, on_progress=None):
    """Decode the encoded position in every time bin.

    `tuning` must already have been through `prepare_tuning`. Chunked only to bound
    memory: the full (n_time_bins, n_positions) score matrix would not fit. This is the
    slowest stage, so it reports its progress.
    """
    activity = gaussian_filter1d(session.spike_counts, config.pv_smooth_bins, axis=0)
    bin_distance_px = _bin_distance_matrix_px(bin_center_x_px, bin_center_y_px)

    decoded_xy_px = np.full((session.n_bins, 2), np.nan)
    peak_score = np.full(session.n_bins, np.nan)

    starts = range(0, session.n_bins, chunk_size)
    report_every = max(1, len(starts) // 10)        # ~10 updates, not one per chunk

    for done, start in enumerate(starts, 1):
        chunk = slice(start, start + chunk_size)
        scores = decoder_scores(activity[chunk], tuning, config, log_prior)
        decoded_xy_px[chunk], peak_score[chunk] = _thresholded_centroid(
            scores, bin_center_x_px, bin_center_y_px, config, bin_distance_px)

        if on_progress and (done % report_every == 0 or done == len(starts)):
            on_progress(f"decoding {100 * done / len(starts):.0f}%")

    return decoded_xy_px, peak_score


def shuffle_threshold(session, config, tuning, log_prior=None, seed=0) -> float:
    """The score the decoder reaches by chance.

    Rotating each cell's spike train in time keeps its rate and rhythmicity but destroys
    its relationship to place and to the other cells. Returns the 99th percentile of the
    resulting scores -- a correlation for `decoder="pv"`, a posterior for `"bayes"`.
    """
    rng = np.random.default_rng(seed)
    shifted = np.stack([np.roll(session.spike_counts[:, unit], rng.integers(session.n_bins))
                        for unit in range(session.n_units)], 1)

    n_samples = min(config.n_shuffle_bins, session.n_bins)
    scores = decoder_scores(
        gaussian_filter1d(shifted[:n_samples], config.pv_smooth_bins, axis=0),
        tuning, config, log_prior)
    return float(np.percentile(scores, config.shuffle_percentile))


In [ ]:
sh99 = shuffle_threshold(sess, cfg, tun, lp)
dec, peak = decode(sess, cfg, tun, bx, by, lp, on_progress=lambda m: print(m, end="\r"))
err = np.hypot(dec[:,0]-sess.track_x_px, dec[:,1]-sess.track_y_px)/cfg.px_per_cm
print(f"\nshuffle99={sh99:.4g} | decoded {np.isfinite(dec[:,0]).mean()*100:.0f}% of bins | median err {np.nanmedian(err):.0f} cm")
w0,w1=5000,5600; fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot(sess.track_x_px[w0:w1], sess.track_y_px[w0:w1], "k", lw=1.5, label="true")
ax[0].scatter(dec[w0:w1,0], dec[w0:w1,1], c=np.arange(w1-w0), cmap="cool", s=12, label="decoded")
ax[0].invert_yaxis(); ax[0].set_aspect("equal"); ax[0].legend(); ax[0].set_title("true vs decoded (6 s)")
ax[1].hist(err[np.isfinite(err)], bins=60, range=(0,300)); ax[1].set_title("decode error (cm)")
plt.tight_layout(); plt.show()


## 7. `lowpass_trajectory` — the anchor

A slow, honest estimate of where the animal really is, that sweeps are measured *against*. In the first
~40 ms of each cycle the sweep hasn't departed, so decoding just those bins recovers the true position;
do that per cycle, smooth across cycles, interpolate to every bin. A sweep = **decoded − anchor**.

In [ ]:
from __future__ import annotations
# ==== DEFINE: lowpass_trajectory   (this is the real code — edit + re-run to change the pipeline) ====

def lowpass_trajectory(session, config, tuning, bin_center_x_px, bin_center_y_px,
                       cycle_onsets, log_prior=None) -> np.ndarray:
    """The anchor: a slowly-moving decoded position that sweeps depart from.

    Decoded from the first 40 ms of each theta cycle, before the sweep has travelled,
    then smoothed across cycles and interpolated back onto every time bin. Sweeps are
    measured against this rather than the tracked position, because the encoded and
    actual positions can drift apart.
    """
    # --- one spike-count vector per cycle, from its first few bins -------------
    window = np.clip(cycle_onsets[:, None] + np.arange(config.anchor_n_bins)[None, :],
                     0, session.n_bins - 1)
    cycle_spike_counts = session.spike_counts[window].sum(1)

    # Smooth across cycles, not time bins: neighbouring cycles see similar places.
    cycle_spike_counts = gaussian_filter1d(cycle_spike_counts.astype(float),
                                           config.anchor_smooth_cycles, axis=0)

    cycle_xy_px, _ = _thresholded_centroid(
        decoder_scores(cycle_spike_counts, tuning, config, log_prior),
        bin_center_x_px, bin_center_y_px, config)

    # Fill unmatched cycles before smoothing, or one NaN spreads across its neighbours.
    fill_value = np.nanmean(cycle_xy_px, 0)
    cycle_xy_px = np.where(np.isfinite(cycle_xy_px), cycle_xy_px, fill_value)
    cycle_xy_px = gaussian_filter1d(cycle_xy_px, config.anchor_post_smooth_cycles, axis=0)

    # --- back onto the time-bin grid, timestamped mid-window -------------------
    cycle_t_s = session.bin_centers_s[cycle_onsets] + 0.5 * config.bin_s * config.anchor_n_bins
    return np.stack([np.interp(session.bin_centers_s, cycle_t_s, cycle_xy_px[:, 0]),
                     np.interp(session.bin_centers_s, cycle_t_s, cycle_xy_px[:, 1])], 1)


In [ ]:
low = lowpass_trajectory(sess, cfg, tun, bx, by, onsets, lp)
print("anchor", low.shape)


## 8. `extract_sweeps` — find the sweeps

Per theta cycle: **grow the longest smooth run** around the moment of peak population firing — points that
don't jump > **20 cm** or turn > **90°** (`_smoothness_breaks` finds the breaks) — **trim the fold-back
tail**, then measure the sweep from the **anchor to the furthest decoded point**. Accept only if ≥ **4
points**, **straight** (r² > 0.5), the animal is **running** (> 15 cm/s), and the decode **beat the
shuffle** (`_mark_unreliable_bins` blanks untrusted bins first). **Every gate is here** — edit this to
change what counts as a sweep.

In [ ]:
from __future__ import annotations
# ==== DEFINE: _wrap_angle, _smoothness_breaks, _mark_unreliable_bins, extract_sweeps   (this is the real code — edit + re-run to change the pipeline) ====

def _wrap_angle(angle):
    """Fold an angle into [-pi, pi)."""
    return (angle + np.pi) % (2 * np.pi) - np.pi


def _smoothness_breaks(decoded, config):
    """Per bin: was the step to the next too long, the step from the previous too long,
    or did the direction turn too sharply. A sweep may not cross any of these."""
    step = np.diff(decoded, axis=0)

    step_to_next_px = np.r_[np.hypot(step[:, 0], step[:, 1]), np.nan]
    step_from_prev_px = np.r_[np.nan, step_to_next_px[:-1]]

    travel_direction = np.arctan2(step[:, 1], step[:, 0])
    turn_angle = np.r_[0.0, np.abs((np.diff(travel_direction) + np.pi) % (2 * np.pi) - np.pi), 0.0]
    is_sharp_turn = turn_angle > config.turn_max_rad

    return step_to_next_px, step_from_prev_px, is_sharp_turn


def _mark_unreliable_bins(session, config, peak_score, lowpass_error_px, shuffle_99):
    """Which decoded time bins to discard. See `Config` for the thresholds."""
    n_active_cells = (session.spike_counts > 0).sum(1)

    too_few_cells = n_active_cells < config.min_active_cells
    decoder_lost_the_animal = lowpass_error_px > config.px(config.max_lowpass_error_cm)

    # A posterior probability is never negative, so the hippocampal branch's floor of 0
    # would be a gate that never fires -- i.e. silently no gate at all. Bayesian decoding
    # therefore always measures its peak against the shuffled null instead.
    if config.decoder == "bayes" or config.min_peak_correlation is None:
        poor_match = peak_score < shuffle_99
    else:
        poor_match = peak_score < config.min_peak_correlation

    return too_few_cells | decoder_lost_the_animal | poor_match


def extract_sweeps(session, config, decoded_xy_px, peak_correlation,
                   lowpass_xy_px, cycle_onsets, shuffle_99) -> dict:
    """Pull one candidate sweep out of each theta cycle.

    Within a cycle: find the bin of peak population firing, grow outwards from it for as
    long as the decoded trajectory stays smooth, then take the stretch from the point
    nearest the anchor to the point furthest from it, and measure its length, direction
    and straightness.

    Returns:
        dict of arrays with one entry per theta cycle. `is_sweep` masks the cycles that
        passed; `path_xy_px` and `path_frame_px` hold the trajectories.
    """
    n_bins = session.n_bins

    # The peak of population firing marks the middle of the sweep.
    population_rate = gaussian_filter1d(session.spike_counts.sum(1).astype(float), 1.0)

    lowpass_error_px = np.hypot(lowpass_xy_px[:, 0] - session.track_x_px,
                                lowpass_xy_px[:, 1] - session.track_y_px)

    # --- blank out the decoded positions we do not trust -----------------------
    decoded = gaussian_filter1d(decoded_xy_px, config.decoded_smooth_bins, axis=0)
    lowpass_smoothed = gaussian_filter1d(lowpass_xy_px, config.lowpass_smooth_bins, axis=0)

    decoded[_mark_unreliable_bins(session, config, peak_correlation,
                                  lowpass_error_px, shuffle_99)] = np.nan

    # --- work relative to the anchor -------------------------------------------
    sweep_frame = decoded - lowpass_smoothed

    step_to_next_px, step_from_prev_px, is_sharp_turn = _smoothness_breaks(decoded, config)
    jump_max_px = config.px(config.jump_max_cm)

    # --- somewhere to put the answers ------------------------------------------
    n_cycles = len(cycle_onsets)
    cycle_centers = np.clip((cycle_onsets + np.r_[cycle_onsets[1:], n_bins]) // 2, 0, n_bins - 1)

    sweeps = dict(
        n_valid_samples=np.zeros(n_cycles, int),
        straightness=np.full(n_cycles, np.nan),
        length_px=np.full(n_cycles, np.nan),
        direction=np.full(n_cycles, np.nan),
        origin_error_px=np.full(n_cycles, np.nan),   # near end of the sweep, to the animal
        speed_px_s=session.speed_px_s[cycle_centers],
        head_direction=session.head_direction[cycle_centers],
        true_xy_px=np.full((n_cycles, 2), np.nan),
        path_xy_px=[None] * n_cycles,           # trajectory in maze coordinates
        path_frame_px=[None] * n_cycles,        # trajectory relative to the anchor
        start_bin=np.full(n_cycles, -1, int),   # chunkThetaPosSweeps.m: s.iStart
        stop_bin=np.full(n_cycles, -1, int),    # chunkThetaPosSweeps.m: s.iStop
        cycle_onsets=cycle_onsets,
    )

    for cycle in range(n_cycles):
        cycle_start = cycle_onsets[cycle]
        cycle_stop = cycle_onsets[cycle + 1] if cycle + 1 < n_cycles else n_bins
        cycle_bins = np.arange(cycle_start, min(cycle_stop, n_bins))
        if len(cycle_bins) < 3:
            continue

        path = decoded[cycle_bins]
        path_frame = sweep_frame[cycle_bins]
        is_finite = np.isfinite(path[:, 0])
        if not is_finite.any():
            continue

        # --- the smooth stretch containing the peak of population firing --------
        # Note: the stretch containing that peak, not the longest one in the cycle.
        peak_activity = int(np.nanargmax(population_rate[cycle_bins]))

        breaks_after = np.where((step_to_next_px[cycle_bins] > jump_max_px)
                                | is_sharp_turn[cycle_bins] | ~is_finite)[0]
        breaks_before = np.where((step_from_prev_px[cycle_bins] > jump_max_px)
                                 | is_sharp_turn[cycle_bins] | ~is_finite)[0]

        run_start = breaks_before[breaks_before < peak_activity].max() \
            if np.any(breaks_before < peak_activity) else 0
        run_stop = breaks_after[breaks_after >= peak_activity].min() \
            if np.any(breaks_after >= peak_activity) else len(cycle_bins) - 1

        # --- from nearest the anchor to furthest from it ------------------------
        masked_frame = path_frame.copy()
        masked_frame[:run_start] = np.nan
        masked_frame[run_stop + 1:] = np.nan

        distance_from_anchor_px = np.hypot(masked_frame[:, 0], masked_frame[:, 1])
        if not np.isfinite(distance_from_anchor_px).any():
            continue

        distal_index = int(np.nanargmax(distance_from_anchor_px))
        proximal_index = int(np.nanargmin(distance_from_anchor_px))
        if proximal_index > distal_index:
            proximal_index = 0          # heading back inwards; take the whole stretch

        sweeps["n_valid_samples"][cycle] = distal_index - proximal_index + 1

        # --- length: greatest separation between any two of its points ----------
        valid_points = path[run_start:run_stop + 1]
        valid_points = valid_points[np.isfinite(valid_points[:, 0])]
        if len(valid_points) >= 2:
            sweeps["length_px"][cycle] = cdist(valid_points, valid_points).max()

        # --- direction: towards its furthest point from the anchor --------------
        sweep_vector_px = masked_frame[distal_index]
        if not np.isfinite(sweep_vector_px).all() or np.hypot(*sweep_vector_px) < 1e-9:
            continue
        sweeps["direction"][cycle] = np.arctan2(sweep_vector_px[1], sweep_vector_px[0])

        sweep_slice = slice(proximal_index, distal_index + 1)
        sweeps["path_xy_px"][cycle] = path[sweep_slice]
        sweeps["path_frame_px"][cycle] = masked_frame[sweep_slice]
        sweeps["start_bin"][cycle] = cycle_start + proximal_index
        sweeps["stop_bin"][cycle] = cycle_start + distal_index
        sweeps["true_xy_px"][cycle] = (session.track_x_px[cycle_start + proximal_index],
                                       session.track_y_px[cycle_start + proximal_index])

        # How far the near end of the sweep is from the animal itself. A sweep is supposed
        # to set off from where the animal is; one that never comes near it is not a sweep.
        sweeps["origin_error_px"][cycle] = np.hypot(
            *(path[proximal_index] - sweeps["true_xy_px"][cycle]))

        # --- straightness: variance along the sweep's own axis / total variance --
        # Equals the paper's r^2 = 1 - var(perpendicular) / var(total). The furthest
        # point is excluded because it defines the axis.
        body_points = masked_frame[proximal_index:distal_index]
        body_points = body_points[np.isfinite(body_points[:, 0])]
        if len(body_points) < 2:
            continue

        sweep_axis = sweep_vector_px / np.hypot(*sweep_vector_px)
        total_variance = body_points[:, 0].var() + body_points[:, 1].var()
        if total_variance > 0:
            sweeps["straightness"][cycle] = (body_points @ sweep_axis).var() / total_variance

    # --- which cycles actually contain a sweep ---------------------------------
    sweeps["is_running"] = sweeps["speed_px_s"] > config.px(config.speed_sweep_cm_s)
    sweeps["starts_at_animal"] = sweeps["origin_error_px"] <= config.px(config.max_sweep_origin_cm)

    # A sweep should point ahead of the animal; the head-centred angle says how far off the
    # head direction it is. Off by default (see Config.max_sweep_head_angle_deg).
    head_centred = _wrap_angle(sweeps["direction"] - sweeps["head_direction"])
    if config.max_sweep_head_angle_deg is None:
        sweeps["points_forward"] = np.ones(n_cycles, bool)
    else:
        sweeps["points_forward"] = np.abs(head_centred) <= np.radians(config.max_sweep_head_angle_deg)

    sweeps["is_sweep"] = (sweeps["n_valid_samples"] >= config.min_valid_samples) \
        & (sweeps["straightness"] > config.straightness_min) \
        & sweeps["is_running"] \
        & sweeps["starts_at_animal"] \
        & sweeps["points_forward"]

    sweeps["prevalence"] = sweeps["is_sweep"].sum() / max(sweeps["is_running"].sum(), 1)
    return sweeps


In [ ]:
sw = extract_sweeps(sess, cfg, dec, peak, low, onsets, sh99)
m = sw["is_sweep"]
print(f"sweeps {int(m.sum())}/{len(m)} | prevalence {sw['prevalence']:.3f} | mean len {np.nanmean(sw['length_px'][m])/cfg.px_per_cm:.1f} cm")


## 9. `alternation` — do sweeps alternate left/right?

Find every run of **three consecutive cycles that all contain a sweep**, label each **left/right of
heading**, and measure how often the direction **flips** across the triplet, vs a shuffle. Above the
shuffle = alternating; here they aren't.

In [ ]:
from __future__ import annotations
# ==== DEFINE: head_centred_direction, alternation   (this is the real code — edit + re-run to change the pipeline) ====

def head_centred_direction(sweeps) -> np.ndarray: # consider to be fixed when dlc_coordinates included
    """Each sweep's direction relative to the animal's heading; positive is left.

    Tracking is in image pixels, whose y axis points DOWN, so angles there run
    clockwise and a leftward sweep gives a negative difference. Negating restores a
    right-handed, left-positive angle. Alternation counts sign flips and is therefore
    unaffected by this; only left/right labels and figures are.
    """
    ego = -_wrap_angle(sweeps["direction"] - sweeps["head_direction"])
    ego[~sweeps["is_sweep"]] = np.nan
    return ego


def alternation(sweeps, n_shuffle=1000, seed=0):
    """Fraction of consecutive sweep triplets that go left-right-left, or the reverse.

    Returns:
        (observed, shuffle_mean, shuffle_99.9th_percentile, n_triplets), where
        n_triplets counts runs of three adjacent theta cycles that all contain a sweep.
        If it is zero the statistic is undefined, not zero; use
        `alternation_consecutive_sweeps` instead.
    """
    is_sweep = sweeps["is_sweep"]
    n_triplets = int(np.sum(is_sweep[:-2] & is_sweep[1:-1] & is_sweep[2:]))
    ego = head_centred_direction(sweeps)

    def alternating_fraction(angles):
        # Alternation means the sign of the turn keeps flipping, so consecutive signs
        # differ by 2 (from +1 to -1, or back).
        turn_sign = np.sign(_wrap_angle(np.diff(angles)))
        sign_change = np.diff(turn_sign)
        usable = np.isfinite(sign_change)
        if usable.sum() == 0:
            return np.nan
        return (np.abs(sign_change[usable]) == 2).sum() / usable.sum()

    observed = alternating_fraction(ego)

    # Null: same directions, same cycles, reordered. Any alternation left is a property
    # of the numbers rather than of their sequence.
    rng = np.random.default_rng(seed)
    present = np.where(np.isfinite(ego))[0]
    null = np.empty(n_shuffle)
    for k in range(n_shuffle):
        shuffled = ego.copy()
        shuffled[present] = ego[present][rng.permutation(len(present))]
        null[k] = alternating_fraction(shuffled)

    return observed, float(np.nanmean(null)), float(np.nanpercentile(null, 99.9)), n_triplets


In [ ]:
obs, nm, nh, ntrip = alternation(sw)
print(f"triplets {ntrip}", f"| observed {obs*100:.1f}% vs shuffle {nm*100:.1f}% (99.9th {nh*100:.1f}%)" if ntrip else "| undefined")


## 10. Plot — single sweeps and runs
`plot_all_sweeps` groups sweeps into **runs** (several consecutive-cycle sweeps per window, like Vollan Fig. 1).

In [ ]:
result = Result(session=sess, config=cfg, sweeps=sw, stats={},
                decoded_xy_px=dec, lowpass_xy_px=low, cycle_onsets=onsets)
plot_sweeps(result); plt.show()
plot_all_sweeps(result, window_s=4.0, ncols=5); plt.show()


## Play around
Edit any **DEFINE** cell and re-run it, then re-run the stages below. Ideas: widen the theta band in
`_theta_phase_from_lfp`; swap `_thresholded_centroid` for an argmax; loosen the gates in `extract_sweeps`.
`io.close()` frees the NWB.

In [ ]:
io.close()
